# Olist AI Logistics — All APIs in One Google Colab

This notebook is a **single end-to-end API demo** using your uploaded `archive.zip` directly. No Kaggle API is used.

It starts the FastAPI service inside Colab and connects:

`Olist real data → AI seller selection → Frenet live carrier quotes → optional Google road distance → product recommendations → Resend customer email`

### Required external credentials

- **Frenet**: `FRENET_TOKEN` and `FRENET_PARTNER_TOKEN` for live carrier quotes. Frenet's authentication documentation says integrations require both the client token and Partner Token; production Partner Token is provided after the integration/homologation process. citeturn761329search1turn761329search0
- **Resend**: `RESEND_API_KEY` and `EMAIL_FROM`. Resend requires an API key; for production sending, verify the sending domain. citeturn417159search4turn761329search5
- **Google Routes**: `GOOGLE_ROUTES_API_KEY` is optional. Routes API uses an API key and provides `computeRouteMatrix` for route distance/duration. citeturn470746search3turn470746search0

### Important data limitation

The public Olist customer/seller files provide ZIP prefixes rather than full 8-digit Brazilian CEPs, and the customer table does not contain customer emails. Therefore the live Frenet quote requires you to supply the full seller/customer CEPs from your real OMS/CRM, and email requires an authorized recipient email from your CRM or a test email you control.


In [ ]:
!pip -q install -U fastapi uvicorn requests pydantic pandas numpy scikit-learn catboost joblib nest-asyncio


## 1. Upload `archive.zip`

Upload the Olist ZIP you provided. The notebook will extract the nine CSV files.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os, json, subprocess, time

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload archive.zip')

zip_name = next(iter(uploaded))
if not zip_name.lower().endswith('.zip'):
    raise ValueError('Please upload archive.zip')

DATA_DIR = Path('/content/olist_real')
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)

with zipfile.ZipFile(zip_name) as z:
    z.extractall(DATA_DIR)

required = [
    'olist_customers_dataset.csv',
    'olist_geolocation_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'product_category_name_translation.csv',
]

for name in required:
    matches = list(DATA_DIR.rglob(name))
    if not matches:
        raise FileNotFoundError(name)

DATA_DIR = next(DATA_DIR.rglob('olist_orders_dataset.csv')).parent
print('Using real Olist data:', DATA_DIR)



## 2. Load the real data and display its size


In [ ]:
import pandas as pd
orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
geo = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

print('orders      :', orders.shape)
print('order items :', items.shape)
print('customers   :', customers.shape)
print('products    :', products.shape)
print('sellers     :', sellers.shape)
print('geolocation :', geo.shape)
display(orders.head(3))


## 3. Train the AI models on the real archive

This trains the repository's freight-cost and late-delivery models and stores the model artifacts where the FastAPI service can load them.


In [ ]:
REPO = Path('/content/Executive-Sales-Analytics-Dashboard')
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run([
    'git','clone','-b','ai-freight-delivery','--depth','1',
    'https://github.com/maroamratef/Executive-Sales-Analytics-Dashboard.git',
    str(REPO)
], check=True)

AI_OUTPUT = REPO / 'ai_output'
AI_OUTPUT.mkdir(parents=True, exist_ok=True)

train = subprocess.run([
    'python', str(REPO / 'python' / 'ai_freight_delivery.py'),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(AI_OUTPUT),
], text=True, capture_output=True)

print(train.stdout)
if train.returncode:
    print(train.stderr)
    raise RuntimeError('AI training failed.')


In [ ]:
summary = json.loads((AI_OUTPUT / 'model_summary.json').read_text())
print(json.dumps(summary, indent=2))


## 4. Add the API credentials as Colab Secrets

Use the **key icon / Secrets** panel in Colab. Create these secrets:

```text
FRENET_TOKEN
FRENET_PARTNER_TOKEN
RESEND_API_KEY
EMAIL_FROM
GOOGLE_ROUTES_API_KEY
APP_API_KEY
```

`APP_API_KEY` is your own protection for the FastAPI endpoints. It is not an external provider credential.

Do not paste API keys into the notebook source or GitHub.


In [ ]:
from google.colab import userdata

def secret(name):
    try:
        value = userdata.get(name)
        return value or ''
    except Exception:
        return ''

os.environ['DATA_DIR'] = str(DATA_DIR)
os.environ['AI_OUTPUT_DIR'] = str(AI_OUTPUT)
os.environ['CUSTOMER_EMAILS_FILE'] = str(REPO / 'config' / 'customer_emails.csv')
os.environ['FRENET_TOKEN'] = secret('FRENET_TOKEN')
os.environ['FRENET_PARTNER_TOKEN'] = secret('FRENET_PARTNER_TOKEN')
os.environ['RESEND_API_KEY'] = secret('RESEND_API_KEY')
os.environ['EMAIL_FROM'] = secret('EMAIL_FROM')
os.environ['GOOGLE_ROUTES_API_KEY'] = secret('GOOGLE_ROUTES_API_KEY')
os.environ['APP_API_KEY'] = secret('APP_API_KEY')

print({
    'FRENET_TOKEN': bool(os.environ['FRENET_TOKEN']),
    'FRENET_PARTNER_TOKEN': bool(os.environ['FRENET_PARTNER_TOKEN']),
    'RESEND_API_KEY': bool(os.environ['RESEND_API_KEY']),
    'EMAIL_FROM': bool(os.environ['EMAIL_FROM']),
    'GOOGLE_ROUTES_API_KEY': bool(os.environ['GOOGLE_ROUTES_API_KEY']),
    'APP_API_KEY': bool(os.environ['APP_API_KEY']),
})


## 5. Start the FastAPI service inside this same Colab

This starts `api/main.py` from the GitHub project. The API reads the trained model files and the real Olist CSVs directly.


In [ ]:
api_log = open('/content/fastapi.log','w')
api_process = subprocess.Popen([
    'uvicorn','api.main:app',
    '--host','127.0.0.1',
    '--port','8000'
], cwd=str(REPO), stdout=api_log, stderr=api_log)

time.sleep(5)
print('FastAPI PID:', api_process.pid)


In [ ]:
import requests

base_url = 'http://127.0.0.1:8000'
headers = {'X-API-Key': os.environ.get('APP_API_KEY','')}

health = requests.get(base_url + '/health', headers=headers, timeout=20)
print('HTTP:', health.status_code)
print(health.json())


## 6. API: product recommendations


In [ ]:
customer_id = str(orders.customer_id.dropna().iloc[0])
rec = requests.get(
    f'{base_url}/recommendations/{customer_id}?limit=5',
    headers=headers,
    timeout=60,
)
print(rec.status_code)
display(pd.DataFrame(rec.json().get('recommendations', [])))


## 7. API: automatically select the nearest/most suitable seller

The API uses real Olist seller/customer/product/geolocation data. It first limits sellers to sellers that historically handled the product basket, then combines geographic proximity, predicted freight, and predicted late risk.

For a live Frenet quote, you must provide **real 8-digit seller and customer CEPs** from your OMS/CRM. The Olist public dataset only provides ZIP prefixes.


In [ ]:
order_id = str(orders.order_id.dropna().iloc[0])
route_demo = requests.post(
    f'{base_url}/orders/{order_id}/route',
    headers=headers,
    json={},
    timeout=120,
)
print('HTTP:', route_demo.status_code)
route_payload = route_demo.json()
print(json.dumps({k:v for k,v in route_payload.items() if k not in ['recommendations','carrier_options']}, indent=2))


## 8. API: live shipping-company selection through Frenet

Frenet's quote endpoint is `POST http://api.frenet.com.br/shipping/quote`. The documented request requires `SellerCEP`, `RecipientCEP`, `ShipmentInvoiceValue`, and `ShippingItemArray`, authenticated with the client `token`. citeturn470746search2turn761329search1

Enter one full seller CEP and one full customer CEP that correspond to the real order for the live test. Do not use a fake zero-padded CEP in production.


In [ ]:
SELLER_CEP = input('Enter FULL 8-digit seller CEP for this test order: ').strip()
RECIPIENT_CEP = input('Enter FULL 8-digit customer CEP for this test order: ').strip()

if len(''.join(ch for ch in SELLER_CEP if ch.isdigit())) != 8 or len(''.join(ch for ch in RECIPIENT_CEP if ch.isdigit())) != 8:
    raise ValueError('Both CEPs must contain 8 digits.')

live_route = requests.post(
    f'{base_url}/orders/{order_id}/route',
    headers=headers,
    json={
        'seller_cep': SELLER_CEP,
        'recipient_cep': RECIPIENT_CEP,
    },
    timeout=120,
)
print('HTTP:', live_route.status_code)
print(json.dumps(live_route.json(), indent=2, default=str)[:12000])


## 9. API: optional Google road routing

Google Routes API provides `computeRouteMatrix` for distances/durations between multiple origins and destinations. The API key must belong to a Google Cloud project with the Routes API enabled. citeturn470746search0turn470746search5


In [ ]:
# Get two coordinates from the selected seller and customer ZIP centroids.
# This cell is optional; exact road routing is separate from the seller AI ranking.

selected_seller_id = route_payload.get('seller_id')
order_row = orders.loc[orders.order_id.eq(order_id)].iloc[0]
customer_row = customers.loc[customers.customer_id.eq(order_row.customer_id)].iloc[0]
seller_row = sellers.loc[sellers.seller_id.eq(selected_seller_id)].iloc[0]

customer_point = {
    'latitude': float(customer_row.customer_lat),
    'longitude': float(customer_row.customer_lon),
}
seller_point = {
    'latitude': float(seller_row.seller_lat),
    'longitude': float(seller_row.seller_lon),
}

matrix = requests.post(
    base_url + '/routes/matrix',
    headers=headers,
    json={
        'origins': [customer_point],
        'destinations': [seller_point],
        'traffic_aware': True,
    },
    timeout=60,
)
print('HTTP:', matrix.status_code)
print(matrix.text[:5000])


## 10. API: generate the personalized customer email

The email contains the selected seller, shipping company/service when a live quote is available, shipping price/ETA, and product recommendations.

The Olist public customer table does not include email addresses. Supply an authorized CRM email as `to_email`, or create `config/customer_emails.csv` with:

```text
customer_id,email
```

Resend's API uses an API key; its current docs recommend creating a key and verifying a sending domain for production. citeturn417159search11turn761329search5


In [ ]:
TEST_EMAIL = input('Enter YOUR verified test email address (not a customer list): ').strip()

email_preview = requests.post(
    f'{base_url}/orders/{order_id}/automate',
    headers=headers,
    json={
        'seller_cep': SELLER_CEP,
        'recipient_cep': RECIPIENT_CEP,
        'email': TEST_EMAIL,
        'send_email': False,
    },
    timeout=120,
)
print('HTTP:', email_preview.status_code)
email_payload = email_preview.json()
print(json.dumps(email_payload, indent=2, default=str)[:12000])


### Send the actual test email

Keep this as a separate explicit cell. It sends only to the single address you enter above.


In [ ]:
SEND_TEST_EMAIL = False

if SEND_TEST_EMAIL:
    sent = requests.post(
        f'{base_url}/orders/{order_id}/automate',
        headers=headers,
        json={
            'seller_cep': SELLER_CEP,
            'recipient_cep': RECIPIENT_CEP,
            'email': TEST_EMAIL,
            'send_email': True,
        },
        timeout=120,
    )
    print('HTTP:', sent.status_code)
    print(sent.text[:10000])
else:
    print('DRY RUN — no email sent. Set SEND_TEST_EMAIL = True after verifying the preview.')


## 11. The complete automation API

Once the credentials and private CRM/OMS data are connected, the single endpoint is:

`POST /orders/{order_id}/automate`

It performs:

`order → eligible sellers → geographic seller ranking → freight prediction → late-risk prediction → live Frenet services → carrier ranking → product recommendations → optional Resend email`

The notebook deliberately does **not** invent missing customer emails or full CEPs.


In [ ]:
# Example of the single request body used by the production workflow.
example_body = {
    'seller_cep': SELLER_CEP,
    'recipient_cep': RECIPIENT_CEP,
    'email': TEST_EMAIL,
    'send_email': False,
    'use_google_routes': bool(os.environ.get('GOOGLE_ROUTES_API_KEY')),
}
print(json.dumps(example_body, indent=2))


## 12. Stop the API when finished


In [ ]:
api_process.terminate()
try:
    api_process.wait(timeout=10)
except subprocess.TimeoutExpired:
    api_process.kill()
print('FastAPI stopped.')
